# Libraries and Dataset


In [1]:
import torch
import torchvision.datasets as dts
from torchvision.transforms import ToTensor
import torchvision.transforms.functional as Trans_F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import random
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F



import numpy as np
import torch
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, adjusted_rand_score, normalized_mutual_info_score
from scipy.stats import mode


import os
import torch
import matplotlib.pyplot as plt

        
        
        
import os
import shutil


import os
import cv2

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

device = DEVICE
print(device)

cpu


# CNN model

In [2]:
# CNN model definition for CIFAR-10
class CNN_small(nn.Module):
    def __init__(self):
        super(CNN_small, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 14 * 14, 10)  # Adjusted for MNIST
        
    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))  # 28x28 -> 14x14
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return F.softmax(x, dim=1)
    

class CNN_large(nn.Module):
    def __init__(self):
        super(CNN_large, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc = nn.Linear(128 * 4 * 4, 10)
        
    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2(x), 2))
        x = F.relu(F.max_pool2d(self.conv3(x), 2))
        
        x = x.view(x.size(0), -1)  # Flatten
        
        x = self.fc(x)
        
        return F.softmax(x, dim=1)

class CNN_large_CIFAR100(nn.Module):
    def __init__(self):
        super(CNN_large_CIFAR100, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(512 * 2 * 2, 1024)
        self.fc2 = nn.Linear(1024, 100)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))  # Output: 32x32 -> 16x16
        x = F.relu(F.max_pool2d(self.conv2(x), 2))  # Output: 16x16 -> 8x8
        x = F.relu(F.max_pool2d(self.conv3(x), 2))  # Output: 8x8 -> 4x4
        x = F.relu(F.max_pool2d(self.conv4(x), 2))  # Output: 4x4 -> 2x2

        x = x.view(x.size(0), -1)  # Flatten
        
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return F.softmax(x, dim=1)




# Autoencoders

In [3]:


class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 16, 16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # (batch, 128, 8, 8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1), # (batch, 256, 4, 4)
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 512, kernel_size=3, stride=2, padding=1), # (batch, 512, 2, 2)
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(512 * 2 * 2, 256),  # Latent space
        )

    def forward(self, x):
        return self.encoder(x)
    
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        self.decoder = nn.Sequential(
            nn.Linear(256, 512 * 2 * 2),
            nn.ReLU(),
            nn.Unflatten(1, (512, 2, 2)),
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1), # (batch, 256, 4, 4)
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1), # (batch, 128, 8, 8)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1), # (batch, 64, 16, 16)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 3, kernel_size=3, stride=2, padding=1, output_padding=1), # (batch, 3, 32, 32)
            nn.Sigmoid(),  # Output in range [0, 1]
        )

    def forward(self, x):
        return self.decoder(x)

class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed



# Gradients Utils


In [4]:
def sparsify_gradients(model, top_percent=0.1):
    """
    Retain only the top `top_percent` of gradients in terms of absolute value for each parameter.
    
    Args:
    - model: The neural network model whose gradients are to be modified.
    - top_percent: The fraction of gradients to keep (default is 0.1, which corresponds to the top 10%).
    """
    for param in model.parameters():
        if param.grad is not None:
            # Flatten the gradient tensor to sort and process
            grad_flat = param.grad.view(-1)
            
            # Compute the absolute values of the gradients
            abs_grad = grad_flat.abs()
            
            # Find the threshold for the top `top_percent` of gradients
            k = int((1 - top_percent) * grad_flat.numel())  # Compute index for top 10%
            if k == 0:  # Handle small tensors
                continue
                
            threshold_value = torch.kthvalue(abs_grad, k).values.item()
            
            # Set gradients below the threshold to zero
            mask = abs_grad >= threshold_value
            grad_flat *= mask.float()  # Mask out gradients below threshold
            
            # Reshape the gradient back to its original shape
            param.grad.copy_(grad_flat.view_as(param.grad))

# Loss Utils


In [5]:
def temp_loss(outputs, targets, temperature):
    # Ensure that outputs is a tensor with batch_size x num_classes dimensions
    # Assume that targets are indices of the correct classes (batch_size,)
    
    batch_size = outputs.size(0)
    
    # Get the log probabilities of the correct class for each example
    correct_class_probs = outputs[torch.arange(batch_size), targets]
    
    # Create a mask for examples where the correct class probability is above the temperature threshold
    mask = correct_class_probs >= temperature
    
    # Compute CrossEntropy loss for each example in the batch
    ce_loss = nn.CrossEntropyLoss(reduction='none')(outputs, targets)
    
    # Zero out the loss for the examples that meet the threshold condition
    ce_loss = ce_loss * (~mask)  # mask is 1D, so no need for unsqueeze
    
    # Compute the mean loss for the batch
    final_loss = ce_loss.mean()
    
    return final_loss



def negatives_loss(outputs, targets):
    # Find the predicted outputs with maximum probability
    max_outputs = torch.argmax(outputs, dim=1)
    
    # Create a mask for correct predictions
    correct_mask = max_outputs == targets
    
    # Compute CrossEntropy loss for each example in the batch
    ce_loss = nn.CrossEntropyLoss(reduction='none')(outputs, targets)
    
    # Select only the loss for incorrect predictions using the mask
    pruned_loss = ce_loss[~correct_mask]
    
    # Compute the mean loss for the incorrect predictions
    if pruned_loss.numel() > 0:
        final_loss = pruned_loss.mean()
    else:
        final_loss = torch.tensor(0.0, device=outputs.device)
   
    loss_list = []
    # For all the losses in the batch
    for loss in ce_loss:
        loss_list.append(loss.item())
    
    return final_loss, loss_list

def half_negatives_loss(outputs, targets):
    ce_loss = nn.CrossEntropyLoss(reduction='none')(outputs, targets)
    
    # Find the indexes of the top half of the losses
    _, top_half_idx = torch.topk(ce_loss, k=ce_loss.size(0) // 2)
    
    # Select the top half of the losses
    top_half_loss = ce_loss[top_half_idx]
    
    # Compute the mean loss for the top half of the losses
    final_loss = top_half_loss.mean()
    
    loss_list = []
    # For all the losses in the batch
    for loss in ce_loss:
        loss_list.append(loss.item())
        
    
    return final_loss, loss_list


#Cross Entropy Loss
def CELoss(outputs, targets):
    loss = nn.CrossEntropyLoss(reduction='none')(outputs, targets)
    loss_list = []
    # For all the losses in the batch
    for loss_item in loss:
        loss_list.append(loss_item.item())
        
    loss = loss.mean()
    return loss, loss_list


def hybrid(outputs,targets,curr_epoch,total_epochs):
    if curr_epoch >= int(total_epochs*0.8):
        return CELoss(outputs,targets)
    else:
        return negatives_loss(outputs,targets)
    
def hybrid_encoder(outputs,targets,curr_epoch,total_epochs):
    if curr_epoch >= int(total_epochs*0.8):
        return MSE(outputs,targets)
    else:
        return Half_MSE(outputs,targets)
    
    
def MSE(outputs,targets):
    criterion = nn.MSELoss(reduction='none')
    loss = criterion(outputs, targets).mean(dim=[1, 2, 3])
    
    loss_list = []
    for loss_item in loss:
        loss_list.append(loss_item.item())
        
    loss = loss.mean()
    return loss, loss_list

def Half_MSE(outputs,targets):
    criterion = nn.MSELoss(reduction='none')
    loss = criterion(outputs,targets)
    loss = criterion(outputs, targets).mean(dim=[1, 2, 3])
    
    _, top_half_idx = torch.topk(loss, k=loss.size(0) // 2)
    
    # Select the top half of the losses
    top_half_loss = loss[top_half_idx]
    
    final_loss = top_half_loss.mean()
    
    loss_list = []
    for l in loss:
        loss_list.append(l.item())
        
    return final_loss, loss_list
    
    

# Training Functions

In [19]:
def train(model, train_loader, optimizer, loss_function, temperature=0.2, epochs=20):
    acc_list = []
    loss_list = [] 
    
    for e in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0  # To track the number of correct predictions
        total_samples = 0  # To track the total number of samples

        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.to(device)
            target = target.to(device)
            
            optimizer.zero_grad()
            output = model(data)

            # Count the number of correct predictions
            truly_correct = (output.argmax(dim=1) == target).float()  # Shape: [batch_size]
            total_correct += truly_correct.sum().item()
            total_samples += data.size(0)
            
            if(loss_function==hybrid):
                loss, losses_gr = loss_function(output, target,e,epochs)
            else:
                loss, losses_gr = loss_function(output, target)

                
            if(batch_idx == 0):
                # Sort losses in descending order
                losses_gr.sort(reverse=True)
                loss_list.append(losses_gr)
                # # Create a histogram
                # plt.plot(range(len(losses_gr)), losses_gr)

                # # Set labels and title
                # plt.xlabel('Index')
                # plt.ylabel('Value')
                # plt.ylim(0, y_upper)
                # plt.title(f'Epoch {e} Losses')

                # # Save the file with 3-digit epoch and 4-digit batch index
                # filename = f'epoch_{e:04d}.png'
                # plt.savefig(os.path.join(folder_name, filename))
                # plt.close()  # Close the figure to save memory
            
            if(loss.item() != 0):
                loss.backward()  # Backpropagate the loss
                optimizer.step()
    
            total_loss += loss.item()

            if batch_idx % 100 == 0:
                # Calculate and append accuracy every 100 batches
                accuracy = total_correct / total_samples if total_samples > 0 else 0
                print(f'Train Epoch: {e} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f} Accuracy: {accuracy:.4f}, Temperature: {temperature}')

        # Calculate and print overall accuracy for the epoch
        overall_accuracy = total_correct / len(train_loader.dataset)
        overall_loss = total_loss / len(train_loader)
        
        print(f'Epoch {e} finished. Overall Accuracy: {overall_accuracy:.4f}, Overall Loss: {overall_loss:.6f}, Temperature: {temperature}')
        acc_list.append(overall_accuracy)
    
    return acc_list, loss_list




def train_autoencoder(model, train_loader, optimizer, loss_function, temperature=0.2, epochs=20):
    losses_list = [] 
        
    for e in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0  # To track the number of correct predictions
        total_samples = 0  # To track the total number of samples

        for batch_idx, (data, target) in enumerate(train_loader):
            data = data.to(device)
            
            optimizer.zero_grad()
            output = model(data)
          
            if(loss_function==hybrid_encoder):
                loss, losses_gr = loss_function(output, data,e,epochs)
            else:
                loss, losses_gr = loss_function(output, data)
            
             
            if(batch_idx == 0):
                # Sort losses in descending order
                losses_gr.sort(reverse=True)
                losses_list.append(losses_gr)
            
            if(loss.item() != 0):
                loss.backward()  # Backpropagate the loss
                optimizer.step()
    
            total_loss += loss.item()

            if batch_idx % 100 == 0:
                # Calculate and append accuracy every 100 batches
                print(f'Train Epoch: {e} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}, Temperature: {temperature}')

        # Calculate and print overall accuracy for the epoch
        overall_loss = total_loss / len(train_loader)
        
        print(f'Epoch {e} finished. Overall Loss: {overall_loss:.6f}, Temperature: {temperature}')
    
    return losses_list


# Evaluation Function

In [7]:
#Test for robustness to noise
def apply_gaussian_noise(image, mean=0, std=0.1):
    """
    Applies Gaussian noise to an image.
    
    Parameters:
        image (torch.Tensor): Input image tensor.
        mean (float): Mean of the Gaussian noise.
        std (float): Standard deviation of the Gaussian noise.
    
    Returns:
        torch.Tensor: Noisy image tensor.
    """
    noise = torch.randn_like(image) * std + mean
    noisy_image = image + noise
    return noisy_image


# Evaluate the model on the test set
def eval(model,testloader,noise_level,device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images = images.to(device)
            labels = labels.to(device)
            noisy_images = apply_gaussian_noise(images, std=noise_level)
            outputs = model(noisy_images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total



def eval_autoencoder(model,testloader,device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            images = images.to(device)
            outputs = model(images)
            loss = nn.MSELoss(reduction='mean')(outputs,images)
            total_loss += loss.item()
    return total_loss / len(testloader)

# Dataset

In [8]:

# Load the CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load training datasets
train_loader_MNIST = DataLoader(
    datasets.MNIST('../data', train=True, download=True, transform=transform),
    batch_size=256, shuffle=True
)

train_loader_CIFAR10 = DataLoader(
    datasets.CIFAR10('../data', train=True, download=True, transform=transform),
    batch_size=256, shuffle=True
)

train_loader_CIFAR100 = DataLoader(
    datasets.CIFAR100('../data', train=True, download=True, transform=transform),
    batch_size=256, shuffle=True
)


# Load test datasets
test_loader_MNIST = DataLoader(
    datasets.MNIST('../data', train=False, download=True, transform=transform),
    batch_size=1024, shuffle=True
)

test_loader_CIFAR10 = DataLoader(
    datasets.CIFAR10('../data', train=False, download=True, transform=transform),
    batch_size=1024, shuffle=True
)

test_loader_CIFAR100 = DataLoader(
    datasets.CIFAR100('../data', train=False, download=True, transform=transform),
    batch_size=1024, shuffle=True
)


Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


# Training

In [18]:
# model1 = CNN_large().to(device)
# model2 = CNN_large().to(device)
# model3 = CNN_large().to(device)

# optimizer1 = optim.Adam(model1.parameters(), lr=0.0001)
# optimizer2 = optim.Adam(model2.parameters(), lr=0.0001)
# optimizer3 = optim.Adam(model3.parameters(), lr=0.0001)

# loss_function1 = half_negatives_loss
# loss_function2 = negatives_loss
# loss_function3 = CELoss
# acc1, l_list1 = train(model1, train_loader_CIFAR10, optimizer1, loss_function1,0,100)
# acc2, l_list2 = train(model2, train_loader_CIFAR10, optimizer2, loss_function2,0,100)
# acc3, l_list3 = train(model3, train_loader_CIFAR10, optimizer3, loss_function3,0,100)



# folder_name = 'losses_folder'
# if not os.path.exists(folder_name):
#     os.makedirs(folder_name)
# for i in range(len(l_list1)):
#     plt.plot(l_list1[i],label='Half Negatives Loss')
#     plt.plot(l_list2[i],label='Negatives Loss')
#     plt.plot(l_list3[i],label='CE Loss')
#     plt.legend(loc=1)
#     plt.title(f'Epoch {i} Losses')
#     plt.ylim(0, 2.5)
#     plt.xlabel('Index')
#     plt.ylabel('Value')
#     plt.title(f'Epoch {i} Losses')
#     # Save the file with 3-digit epoch and 4-digit batch index
#     filename = f'epoch_{i:04d}.png'
#     plt.savefig(os.path.join(folder_name, filename))
#     plt.close()  # Close the figure to save memory


# eval_acc1 = eval(model1,test_loader_CIFAR10,0,device)
# eval_acc2 = eval(model2,test_loader_CIFAR10,0,device)
# eval_acc3 = eval(model3,test_loader_CIFAR10,0,device)

# plt.plot(acc3, label='CE Loss')
# plt.plot(acc2, label='Hybrid Loss')
# plt.plot(acc1, label='Half Negatives Loss')
# plt.legend()
# plt.title('Training Accuracy')
# plt.show()

# print(f'CE Loss: {eval_acc1:.4f}')
# print(f'Hybrid Loss: {eval_acc2:.4f}')
# print(f'Half Negatives Loss: {eval_acc3:.4f}')



In [20]:
model1 = Autoencoder().to(device)
model2 = Autoencoder().to(device)
model3 = Autoencoder().to(device)

optimizer1 = optim.Adam(model1.parameters(), lr=0.0001)
optimizer2 = optim.Adam(model2.parameters(), lr=0.0001)
optimizer3 = optim.Adam(model3.parameters(), lr=0.0001)

loss_function1 = MSE
loss_function2 = Half_MSE
loss_function3 = hybrid_encoder


l_list1 = train_autoencoder(model1, train_loader_CIFAR10, optimizer1, loss_function1,0,100)
l_list2 = train_autoencoder(model2, train_loader_CIFAR10, optimizer2, loss_function2,0,100)
l_list3 = train_autoencoder(model3, train_loader_CIFAR10, optimizer3, loss_function3,0,100)




folder_name = 'autoencoder_losses_folder'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
for i in range(len(l_list1)):
    plt.plot(l_list1[i],label='MSE Loss')
    plt.plot(l_list2[i],label='Half MSE Loss')
    plt.plot(l_list3[i],label='Hybrid Loss')
    plt.legend(loc=1)
    plt.title(f'Epoch {i} Losses')
    plt.ylim(0, 0.011)
    plt.xlabel('Index')
    plt.ylabel('Value')
    plt.title(f'Epoch {i} Losses')
    # Save the file with 3-digit epoch and 4-digit batch index
    filename = f'epoch_{i:04d}.png'
    plt.savefig(os.path.join(folder_name, filename))
    plt.close()  # Close the figure to save memory



np.save('./autoencoder_losses_folder/MSE_losses.npy',l_list1)
np.save('./autoencoder_losses_folder/Half_MSE_losses.npy',l_list2)
np.save('./autoencoder_losses_folder/Hybrid_losses.npy',l_list3)
# eval_loss1 = eval_autoencoder(model1,test_loader_CIFAR10,device)
# eval_loss2 = eval_autoencoder(model2,test_loader_CIFAR10,device)
# eval_loss3 = eval_autoencoder(model3,test_loader_CIFAR10,device)

# plt.plot(acc1, label='MSE Loss')
# plt.plot(acc2, label='Half MSE Loss')
# plt.plot(acc3, label='Hybrid Loss')
# plt.legend()
# plt.title('Training Loss')
# plt.show()

# print(f'MSE Loss: {eval_loss1:.4f}')
# print(f'Half MSE Loss: {eval_loss2:.4f}')
# print(f'Hybrid Loss: {eval_loss3:.4f}')



Train Epoch: 0 [0/50000] Loss: 0.095823, Temperature: 0
Train Epoch: 0 [25600/50000] Loss: 0.024379, Temperature: 0
Epoch 0 finished. Overall Loss: 0.031112, Temperature: 0
Train Epoch: 1 [0/50000] Loss: 0.016762, Temperature: 0
Train Epoch: 1 [25600/50000] Loss: 0.013137, Temperature: 0
Epoch 1 finished. Overall Loss: 0.014161, Temperature: 0
Train Epoch: 2 [0/50000] Loss: 0.012157, Temperature: 0
Train Epoch: 2 [25600/50000] Loss: 0.011220, Temperature: 0
Epoch 2 finished. Overall Loss: 0.011506, Temperature: 0
Train Epoch: 3 [0/50000] Loss: 0.010231, Temperature: 0
Train Epoch: 3 [25600/50000] Loss: 0.010630, Temperature: 0
Epoch 3 finished. Overall Loss: 0.010077, Temperature: 0
Train Epoch: 4 [0/50000] Loss: 0.009131, Temperature: 0
Train Epoch: 4 [25600/50000] Loss: 0.008777, Temperature: 0
Epoch 4 finished. Overall Loss: 0.009073, Temperature: 0
Train Epoch: 5 [0/50000] Loss: 0.008640, Temperature: 0
Train Epoch: 5 [25600/50000] Loss: 0.008682, Temperature: 0
Epoch 5 finished. O

In [24]:

folder_name = 'autoencoder_losses_folder'
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
for i in range(len(l_list1)):
    plt.plot(l_list1[i],label='MSE Loss')
    plt.plot(l_list2[i],label='Half MSE Loss')
    plt.plot(l_list3[i],label='Hybrid Loss')
    plt.legend(loc=1)
    plt.title(f'Epoch {i} Losses')
    plt.ylim(0, 0.05)
    plt.xlabel('Index')
    plt.ylabel('Value')
    plt.title(f'Epoch {i} Losses')
    # Save the file with 3-digit epoch and 4-digit batch index
    filename = f'epoch_{i:04d}.png'
    plt.savefig(os.path.join(folder_name, filename))
    plt.close()  # Close the figure to save memory

# Video with Activations

In [25]:


## Ensure the output directories exist
# directory = 'videodata'


# tensors_directory = 'videodata_CE'
# # Loop through each example (second dimension) for act3_tensor
# for i in range(50):  # Loop over 50 examples
#     # Extract the 300, 10 tensor for the i-th example
#     example_tensor = torch.load(os.path.join(tensors_directory, f'example_{i+1}.pt'))
#     # Create a subfolder for the histograms of the i-th example
#     hist_dir = os.path.join(directory, f'histogram_example_{i+1}')
#     if(os.path.exists(hist_dir)):
#         continue
#     os.makedirs(hist_dir, exist_ok=True)

#     # Generate and save histograms for each epoch
#     for epoch in range(example_tensor.size(0)):  # 300 epochs
#         plt.figure()
#         plt.bar(range(10), example_tensor[epoch].numpy())  # 10 classes
#         plt.xlabel('Classes')
#         plt.ylabel('Activations')
#         plt.title(f'Epoch {epoch + 1} - Example {i + 1}')
#         plt.xticks(range(10))  # Set x-ticks to correspond to classes
#         plt.ylim(0, example_tensor.max().item())  # Set y-limits for clarity

#         # Save the histogram
#         plt.savefig(os.path.join(hist_dir, f'epoch_{epoch + 1}.png'))
#         plt.close()  # Close the figure to save memory

#     print(f'Histograms for Example {i + 1} saved in {hist_dir}')

# # Create the second output directory for act4_tensor
# os.makedirs('videodata_neg', exist_ok=True)

# # Loop through each example (second dimension) for act4_tensor
# for i in range(50):  # Loop over 50 examples
#     # Extract the 300, 10 tensor for the i-th example
#     example_tensor = torch.load(os.path.join('videodata_neg', f'example_{i+1}.pt'))
    
#     # Create a subfolder for the histograms of the i-th example
#     hist_dir = os.path.join('videodata_neg', f'histogram_example_{i+1}')
    
#     if(os.path.exists(hist_dir)):
#         continue
#     os.makedirs(hist_dir, exist_ok=True)

#     # Generate and save histograms for each epoch
#     for epoch in range(example_tensor.size(0)):  # 300 epochs
#         plt.figure()
#         plt.bar(range(10), example_tensor[epoch].numpy())  # 10 classes
#         plt.xlabel('Classes')
#         plt.ylabel('Activations')
#         plt.title(f'Epoch {epoch + 1} - Example {i + 1}')
#         plt.xticks(range(10))  # Set x-ticks to correspond to classes
#         plt.ylim(0, example_tensor.max().item())  # Set y-limits for clarity

#         # Save the histogram
#         plt.savefig(os.path.join(hist_dir, f'epoch_{epoch + 1}.png'))
#         plt.close()  # Close the figure to save memory

#     print(f'Histograms for Example {i + 1} saved in {hist_dir}')


# Function to generate a video from images in a folder with adjustable fps
def create_video_from_images(image_folder, video_name, fps=5):  # Reduced FPS from 10 to 5
    images = sorted([img for img in os.listdir(image_folder) if img.endswith(".png")])
    
    if len(images) == 0:
        print(f"No images found in {image_folder}")
        return
    
    # Get the size of the first image
    frame = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = frame.shape

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 files
    video = cv2.VideoWriter(video_name, fourcc, fps, (width, height))

    for image in images:
        img_path = os.path.join(image_folder, image)
        frame = cv2.imread(img_path)
        video.write(frame)

    # Release the video writer object
    video.release()
    print(f'Video saved as {video_name}')


if not os.path.exists('loss_dynamics'):
    os.mkdir('loss_dynamics')

# Main loop to go through each 'histogram_example_i' folder and create a video

parent_directory = 'autoencoder_losses_folder'
if os.path.exists(parent_directory):
    video_path = os.path.join(parent_directory, 'Autoencoder_large_scale.mp4')
    create_video_from_images(parent_directory, video_path, fps=5)  # Pass the desired FPS
else:
    print(f'{parent_directory} does not exist.')
    
#Move video to loss_dynamics folder
shutil.move(video_path, os.path.join('loss_dynamics', os.path.basename(video_path)))


        


# # Define the source directory where the .mp4 files are currently located
# source_dir = 'videodata_neg'

# # Define the target directory to move the .mp4 files
# target_dir = os.path.join(source_dir, 'Neg_Loss_videos')

# # Create the target directory if it does not exist
# os.makedirs(target_dir, exist_ok=True)

# # Loop through the source directory and find all .mp4 files
# for file_name in os.listdir(source_dir):
#     if file_name.endswith(".mp4"):
#         # Construct the full file path
#         source_file = os.path.join(source_dir, file_name)
#         target_file = os.path.join(target_dir, file_name)

#         # Move the file to the target directory
#         shutil.move(source_file, target_file)
#         print(f'Moved {file_name} to {target_dir}')

# print("All .mp4 files have been moved to the CE_Loss_videos folder.")


# # Define the source directory where the .mp4 files are currently located
# source_dir = 'videodata'

# # Define the target directory to move the .mp4 files
# target_dir = os.path.join(source_dir, 'CE_Loss_videos')

# # Create the target directory if it does not exist
# os.makedirs(target_dir, exist_ok=True)

# # Loop through the source directory and find all .mp4 files
# for file_name in os.listdir(source_dir):
#     if file_name.endswith(".mp4"):
#         # Construct the full file path
#         source_file = os.path.join(source_dir, file_name)
#         target_file = os.path.join(target_dir, file_name)

#         # Move the file to the target directory
#         shutil.move(source_file, target_file)
#         print(f'Moved {file_name} to {target_dir}')

# print("All .mp4 files have been moved to the CE_Loss_videos folder.")

Video saved as autoencoder_losses_folder/Autoencoder_large_scale.mp4


'loss_dynamics/Autoencoder_large_scale.mp4'

# Evaluation with Noise

In [12]:
# #Evaluate the models for robustness to noise
# noise_levels = [x * 0.01 for x in range(0, 101)]
# noise_accs = []


# for noise_level in noise_levels:
#     noisy_acc1 = eval(model1,test_loader_CIFAR10,noise_level,device)
#     noisy_acc2 = eval(model2,test_loader_CIFAR10,noise_level,device)
#     noisy_acc3 = eval(model3,test_loader_CIFAR10,noise_level,device)
#     noise_accs.append((noisy_acc1, noisy_acc2,noisy_acc3))
    
# plt.plot(noise_levels, [acc1 for acc1, _,__ in noise_accs], label='Half Negatives')
# plt.plot(noise_levels, [acc2 for _, acc2,_ in noise_accs], label='Hybrid')
# plt.plot(noise_levels, [acc3 for _,_, acc3 in noise_accs], label='CE Loss')
# plt.xlabel('Noise Level')
# plt.ylabel('Accuracy')
# plt.title('CIFAR10')
# plt.legend()
# plt.show()



RuntimeError: The size of tensor a (32) must match the size of tensor b (1024) at non-singleton dimension 2